# Lesson 11

## Video 1 — From One Tool to a Toolbox

In [2]:
# Import the three tools and their definitions
from claude_assist.tools import (
    calculate,
    get_time,
    count_words,
    TOOL_DEFINITIONS
)

print("Registered tools:")
for tool in TOOL_DEFINITIONS:
    print("-", tool["name"])

Registered tools:
- calculate
- get_time
- count_words


In [3]:
# Test each Python function directly
print("Calculate:", calculate(27, 18, "multiply"))
print("Time:", get_time())
print("Words:", count_words("Claude can now use several tools"))

Calculate: 486
Time: 14:45
Words: 6


## Video 2 — Building the Dispatcher

In [4]:
# Import the dispatcher
from claude_assist.tools import dispatch

print(dispatch(
    "calculate",
    {"a": 12, "b": 7, "operation": "multiply"}
))

84


In [7]:
# The same dispatcher can route to a different tool
print(dispatch(
    "count_words",
    {"text": "Claude can choose the right tool for each task"}
))

9


In [8]:
# An unknown tool returns a clear error instead of crashing
print(dispatch(
    "missing_tool",
    {}
))

error: unknown tool 'missing_tool'


`**args` passes the values from Claude's input dictionary into the matching Python function.

## Video 3 — Handling Multiple Tool Requests

In [1]:
# Set up the Claude client
from dotenv import load_dotenv
import anthropic

from claude_assist import config

load_dotenv()
client = anthropic.Anthropic()

In [10]:
# Ask one question that needs two different tools
messages = [{
    "role": "user",
    "content": (
        'What is 27 multiplied by 18, and how many words are in '
        '"Claude can now use several tools"? Use the tools.'
    )
}]

resp = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=TOOL_DEFINITIONS,
    messages=messages
)

print("Stop reason:", resp.stop_reason)

for block in resp.content:
    print(block)

Stop reason: tool_use
ToolUseBlock(id='toolu_01JN58dyxhqEcZuxuTxwvbPQ', caller=DirectCaller(type='direct'), input={'a': 27, 'b': 18, 'operation': 'multiply'}, name='calculate', type='tool_use', toolset_name=None)
ToolUseBlock(id='toolu_01HHZt2HTKp4esDFBBE68jsZ', caller=DirectCaller(type='direct'), input={'text': 'Claude can now use several tools'}, name='count_words', type='tool_use', toolset_name=None)


In [11]:
# Run every tool Claude requested and collect the results
results = []

for block in resp.content:
    if block.type == "tool_use":
        result = dispatch(
            block.name,
            block.input
        )

        results.append({
            "type": "tool_result",
            "tool_use_id": block.id,
            "content": str(result)
        })

        print(block.name, "->", result)

calculate -> 486
count_words -> 6


In [12]:
# Keep Claude's tool requests and return all results together
messages.append({
    "role": "assistant",
    "content": resp.content
})

messages.append({
    "role": "user",
    "content": results
})

final = client.messages.create(
    model=config.MODEL,
    max_tokens=config.MAX_TOKENS,
    system=config.SYSTEM,
    tools=TOOL_DEFINITIONS,
    messages=messages
)

print(final.content[0].text)

- 27 × 18 = **486**
- "Claude can now use several tools" has **6** words.


## Video 4 — Shaping Tool Choice and Validation

In [14]:
# Valid input works normally
print(calculate(10, 3, "divide"))

3.3333333333333335


In [15]:
# Valid input works normally
print(calculate(10, 5, "divide"))

# Invalid inputs return clear errors
print(calculate("ten", 5, "add"))
print(calculate(10, 5, "power"))
print(count_words(123))

2.0
error: a must be a number
error: invalid operation
error: text must be a string


The goal is simple: bad inputs should fail clearly, not crash the application.

## Video 5 — Lab Validation and Recap

In [3]:
# Test the full Conversation class with a multi-tool prompt
from claude_assist.conversation import Conversation

convo = Conversation(config.SYSTEM)

reply = convo.send(
    'What is 15 multiplied by 8, and how many words are in '
    '"Tools make Claude more useful"?'
)

print("Claude:", reply)

Claude: - 15 × 8 = **120**
- "Tools make Claude more useful" has **5** words.


In [4]:
# Check that the conversation contains the complete exchange
print("Messages stored:", len(convo.history))

for message in convo.history:
    print(message["role"], "->", message["content"])

Messages stored: 4
user -> What is 15 multiplied by 8, and how many words are in "Tools make Claude more useful"?
assistant -> [ToolUseBlock(id='toolu_01Bh7N9RSUQf7BraSNNFoA3g', caller=DirectCaller(type='direct'), input={'a': 15, 'b': 8, 'operation': 'multiply'}, name='calculate', type='tool_use', toolset_name=None), ToolUseBlock(id='toolu_016q27DTWBZ2VLHhHHyZFvTN', caller=DirectCaller(type='direct'), input={'text': 'Tools make Claude more useful'}, name='count_words', type='tool_use', toolset_name=None)]
user -> [{'type': 'tool_result', 'tool_use_id': 'toolu_01Bh7N9RSUQf7BraSNNFoA3g', 'content': '120'}, {'type': 'tool_result', 'tool_use_id': 'toolu_016q27DTWBZ2VLHhHHyZFvTN', 'content': '5'}]
assistant -> [TextBlock(citations=None, text='- 15 × 8 = **120**\n- "Tools make Claude more useful" has **5** words.', type='text')]
